In [1]:
import pandas as pd
import numpy as np


#Load dataset

In [2]:
df = pd.read_csv('helpdesk_tickets.csv')

In [4]:
df.columns = df.columns.str.lower()
df.columns

Index(['ticket_id', 'date_created', 'date_resolved', 'category', 'subcategory',
       'priority', 'status', 'assigned_team', 'resolution_time_hrs',
       'description', 'escalated'],
      dtype='object')

In [5]:
print(df[['assigned_team', 'resolution_time_hrs', 'escalated']].head())

         assigned_team  resolution_time_hrs  escalated
0      Desktop Support                 3.56       True
1        Security Team                 4.14      False
2        Security Team                 2.98       True
3  Application Support                 4.45      False
4      Desktop Support                 3.49       True


In [6]:
df['resolution_time_hrs'].describe()

count    1000000.000000
mean           5.246152
std            2.742932
min            0.500000
25%            2.870000
50%            5.240000
75%            7.620000
max           10.000000
Name: resolution_time_hrs, dtype: float64

In [7]:
df['resolution_time_hrs'] = (
    pd.to_datetime(df['date_resolved']) - 
    pd.to_datetime(df['date_created'])
).dt.total_seconds() / 3600

In [8]:
df['resolution_time_hrs'].describe()

count    700057.000000
mean         95.965786
std          48.003193
min          24.000000
25%          48.000000
50%          96.000000
75%         144.000000
max         168.000000
Name: resolution_time_hrs, dtype: float64

In [9]:
import numpy as np

In [10]:
df['complexity'] = np.random.choice(
    ['Low', 'Medium', 'High'],
    size=len(df),
    p=[0.5, 0.3, 0.2]
)

In [11]:
df['complexity'].value_counts(normalize=True)

complexity
Low       0.500478
Medium    0.299426
High      0.200096
Name: proportion, dtype: float64

In [12]:
team_factor = {
    'Application Support': 1.0,
    'Desktop Support': 0.8,
    'Network Team': 1.3,
    'Security Team': 1.5
}

df['resolution_time_hrs'] = df['resolution_time_hrs'] * df['assigned_team'].map(team_factor)

In [13]:
df[['assigned_team', 'resolution_time_hrs']].head()

,assigned_team,resolution_time_hrs
0,Desktop Support,134.4
1,Security Team,108.0
2,Security Team,252.0
3,Application Support,120.0
4,Desktop Support,134.4


In [14]:
complexity_factor = {
    'Low': 0.7,
    'Medium': 1.0,
    'High': 1.5
}

df['resolution_time_hrs'] = df['resolution_time_hrs'] * df['complexity'].map(complexity_factor)

In [15]:
df[['assigned_team', 'complexity', 'resolution_time_hrs']].head()

,assigned_team,complexity,resolution_time_hrs
0,Desktop Support,High,201.6
1,Security Team,Low,75.6
2,Security Team,Low,176.4
3,Application Support,Low,84.0
4,Desktop Support,Medium,134.4


# redefining escalation

In [16]:
df['escalated'] = (
    (df['resolution_time_hrs'] > 120) |
    (df['complexity'] == 'High')
)

In [17]:
df[['resolution_time_hrs', 'complexity', 'escalated']].head()

,resolution_time_hrs,complexity,escalated
0,201.6,High,True
1,75.6,Low,False
2,176.4,Low,True
3,84.0,Low,False
4,134.4,Medium,True


In [18]:
metrics = df.groupby('assigned_team').agg(
    ticket_count=('ticket_id', 'count'),
    avg_resolution_time=('resolution_time_hrs', 'mean'),
    median_resolution_time=('resolution_time_hrs', 'median'),
    max_resolution_time=('resolution_time_hrs', 'max'),
    escalation_rate=('escalated', 'mean')
).reset_index()

print(metrics)

         assigned_team  ticket_count  avg_resolution_time  \
0  Application Support        200628            90.818262   
1      Desktop Support        399425            73.089672   
2         Network Team        199943           118.532779   
3        Security Team        200004           136.804218   

   median_resolution_time  max_resolution_time  escalation_rate  
0                    84.0                252.0         0.258339  
1                    67.2                201.6         0.230659  
2                   109.2                327.6         0.420470  
3                   126.0                378.0         0.469341  


#adding randomness

In [19]:
df['escalated'] = (
    (df['resolution_time_hrs'] > 120) |
    (df['complexity'] == 'High') |
    (np.random.rand(len(df)) < 0.05)
)

#remove randomness

In [20]:
df['escalated'] = (
    (df['resolution_time_hrs'] > 120) |
    (df['complexity'] == 'High')
)

In [21]:
metrics = df.groupby('assigned_team').agg(
    ticket_count=('ticket_id', 'count'),
    avg_resolution_time=('resolution_time_hrs', 'mean'),
    median_resolution_time=('resolution_time_hrs', 'median'),
    max_resolution_time=('resolution_time_hrs', 'max'),
    escalation_rate=('escalated', 'mean')
).reset_index()

print(metrics)

         assigned_team  ticket_count  avg_resolution_time  \
0  Application Support        200628            90.818262   
1      Desktop Support        399425            73.089672   
2         Network Team        199943           118.532779   
3        Security Team        200004           136.804218   

   median_resolution_time  max_resolution_time  escalation_rate  
0                    84.0                252.0         0.258339  
1                    67.2                201.6         0.230659  
2                   109.2                327.6         0.420470  
3                   126.0                378.0         0.469341  


#save cleaned dataset

In [22]:
df.to_csv('cleaned_helpdesk_tickets.csv', index=False)